# Hidden Markov Model (HMM) Analysis
### Master Thesis: Revisiting Delegation Theory in the Age of AI: Dynamic Algorithm Appreciation and Aversion in Triadic Organizational Relationships

---

### Table of Contents

**Part I — Data and HMM Estimation Framework**

1. [Data Description and Variable Construction](#1-data-description-and-variable-construction)
   - 1.1 &nbsp; [Helpers & Imports](#11-helpers--imports)
   - 1.2 &nbsp; [Benchmark Construction](#12-benchmark-construction)
   - 1.3 &nbsp; [Data Loading & Inspection](#13-data-loading--inspection)
2. [Hidden Markov Model Specification](#2-hidden-markov-model-specification)
   - 2.1 &nbsp; [Model Structure & Forward–Backward Algorithm](#21-model-structure--22-forwardbackward-algorithm)
   - 2.2 &nbsp; [Maximum Likelihood Estimation](#22-maximum-likelihood-estimation)
3. [Model Selection and Parameter Estimation](#3-model-selection-and-parameter-estimation)
4. [Posterior State Inference and Interpretation](#4-posterior-state-inference-and-interpretation)
   - 4.1 &nbsp; [Posterior State Assignment](#41-posterior-state-assignment)
   - 4.2 &nbsp; [Results Visualisation](#42-results-visualisation)

**Part II — Empirical Findings**

5. [KPI-Driven Transition Dynamics](#5-kpi-driven-transition-dynamics)
6. [Transparency as Moderator of State Transitions](#6-transparency-as-moderator-of-state-transitions)
7. [Strategic Control Retention Under High Task Stakes](#7-strategic-control-retention-under-high-task-stakes)

**Part III — Theoretical and Task-Level Implications**

8. [Delegation as Dynamic Learning Process](#8-delegation-as-dynamic-learning-process)
9. [Authority as Strategic and Legitimacy Mechanism](#9-authority-as-strategic-and-legitimacy-mechanism)
10. [Triadic Delegation Flows: Authority vs Execution](#10-triadic-delegation-flows-authority-vs-execution)
11. [Temporal Evolution of Delegation Patterns](#11-temporal-evolution-of-delegation-patterns)

**Appendix**

12. [Cluster Bootstrap Standard Errors](#12-cluster-bootstrap-standard-errors)
13. [Model Summary Table](#13-model-summary-table)

---
# Part I — Data and HMM Estimation Framework

<a id="1-data-description-and-variable-construction"></a>
## 1. Data Description and Variable Construction
<a id="11-helpers--imports"></a>
### 1.1 Helpers & Imports

In [2]:
# ============================================================
# 1. Imports & Helpers
# ============================================================
from __future__ import annotations

import os
import time
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple
from pathlib import Path
from multiprocessing.pool import ThreadPool

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from scipy.special import logsumexp
from sklearn.preprocessing import StandardScaler

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

# ---- resolve data path ----
MANUAL_XLSX_PATH = None
PREFERRED_DATASETS = [
    Path(r"..\triadic_simulation\data\Triadic_Delegation_Dataset_SYNTH_ANALYSIS.xlsx"),
]
DATA_PATH = None
if MANUAL_XLSX_PATH:
    DATA_PATH = Path(MANUAL_XLSX_PATH)
else:
    for p in PREFERRED_DATASETS:
        if p.exists():
            DATA_PATH = p
            break
assert DATA_PATH is not None and DATA_PATH.exists(), \
    f"Data file not found. Tried: {PREFERRED_DATASETS}"
print(f"Data file: {DATA_PATH.resolve()}")


def softmax(z, axis=-1):
    """Numerically stable softmax (works on 1-D vectors and row-wise on 2-D)."""
    z = z - np.max(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=axis, keepdims=True)


def log_softmax(z, axis=-1):
    """Numerically stable log-softmax."""
    return z - logsumexp(z, axis=axis, keepdims=True)


def log_gaussian_diag(y, mean, log_sigma):
    """Scalar version (kept for reference)."""
    sigma2 = np.exp(2 * log_sigma)
    return -0.5 * (
        np.sum(np.log(2 * np.pi * sigma2))
        + np.sum((y - mean) ** 2 / sigma2)
    )

print("Section 1 — Imports & helpers loaded.")

Data file: C:\Users\Admin\OneDrive\Desktop\Algorithm-Appreciation-and-Aversion-in-Triadic-Delegation-Settings\triadic_simulation\data\Triadic_Delegation_Dataset_SYNTH_ANALYSIS.xlsx
Section 1 — Imports & helpers loaded.


<a id="12-benchmark-construction"></a>
### 1.2 Benchmark Construction
Derive performance benchmarks used as **transition covariates** (P = 4):
- `kpi_operational_gap_index` — composite operational KPIs gap (intermediate)
- `within_unit_temporal_benchmark` — period-over-period KPI change, **mean-centred**
- `horizontal_peer_benchmark` — cross-sectional percentile rank within period
- `threshold_benchmark` — recent negative shock indicator (binary)
- `transparency_level_norm` — transparency main effect (normalised 0–1)

> `within_unit_ai_trajectory_benchmark` trimmed (high VIF). Transparency × performance interactions excluded in this specification.


In [3]:
# ============================================================
# 2. Build Benchmarks
# ============================================================

def build_benchmarks(df):
    df = df.sort_values(["manager_id", "period_id"]).copy()

    # Composite performance index (higher = better)
    df["kpi_operational_gap_index"] = (
        (-1.0) * df["service_level_delta"]
        + (-0.6) * df["inventory_cost_delta"]
        + (-0.4) * df["expedite_cost_delta"]
        + (-1.2) * df["error_incident_count"]
    )

    # Within-manager temporal benchmark (period-over-period diff, mean-centred)
    df["within_unit_temporal_benchmark"] = (
        df.groupby("manager_id")["kpi_operational_gap_index"].diff(1)
    )
    temp_mean = df["within_unit_temporal_benchmark"].mean()
    df["within_unit_temporal_benchmark"] = (
        df["within_unit_temporal_benchmark"] - temp_mean
    )

    # Peer percentile (cross-sectional)
    pct = df.groupby("period_id")["kpi_operational_gap_index"].rank(pct=True)
    df["horizontal_peer_benchmark"] = (pct - 0.5) * 2.0

    # Threshold shock (binary)
    df["threshold_benchmark"] = df["recent_negative_shock"].astype(float)

    # ── Transparency main effect ─────────────────────────────────────────────
    df["transparency_level_norm"] = df["transparency_level"].astype(float) / 3.0

    return df

print("Section 2 — build_benchmarks() defined.")


Section 2 — build_benchmarks() defined.


<a id="13-data-loading--inspection"></a>
### 1.3 Data Loading & Inspection
Load the `panel_manager_period` sheet, build benchmarks, scale variables, and form per-manager sequences.

In [4]:
# ============================================================
# 3. Data Loader
# ============================================================

@dataclass
class HMMData:
    Y: List[np.ndarray]      # emission sequences
    X: List[np.ndarray]      # transition covariates
    Z: List[np.ndarray]      # emission controls
    ids: List[str]
    periods: List[np.ndarray]
    y_scaler: StandardScaler
    x_scaler: StandardScaler
    z_scaler: StandardScaler


def load_sequences(xlsx_path):
    df = pd.read_excel(xlsx_path, sheet_name="panel_manager_period")

    # Safety: analysis file should NOT contain latent truth columns
    forbidden = ["latent_state_true", "latent_state_true_next"]
    if any(c in df.columns for c in forbidden):
        df = df.drop(columns=[c for c in forbidden if c in df.columns])
        print("  Dropped latent truth columns to proceed with analysis data.")

    df = build_benchmarks(df)

    # ── Compute share_authority_esc from decision_episode ──
    dec_ep = pd.read_excel(xlsx_path, sheet_name="decision_episode")
    dec_ep["is_escalated"] = (dec_ep["escalation_flag"] == 1).astype(int)
    esc_agg = (dec_ep
        .groupby(["manager_id", "period_id"])
        .agg(n_tasks=("episode_id", "count"),
             n_escalated=("is_escalated", "sum"))
        .reset_index())
    esc_agg["share_authority_esc"] = esc_agg["n_escalated"] / esc_agg["n_tasks"]
    df = df.merge(esc_agg[["manager_id", "period_id", "share_authority_esc"]],
                  on=["manager_id", "period_id"], how="left")
    df["share_authority_esc"] = df["share_authority_esc"].fillna(0.0)
    print(f"  share_authority_esc computed from decision_episode "
          f"(mean={df['share_authority_esc'].mean():.4f}, "
          f"std={df['share_authority_esc'].std():.4f})")

    # D=2: AI authority share + manager escalation share
    emission_cols = ["ai_decision_authority_share", "share_authority_esc"]

    # P=4: 3 benchmarks + transparency main effect only (no interactions).
    # within_unit_ai_trajectory_benchmark trimmed (high VIF).
    transition_cols = [
        "within_unit_temporal_benchmark",   # mean-centred period-over-period KPI diff
        "horizontal_peer_benchmark",        # cross-sectional percentile rank
        "threshold_benchmark",              # recent negative shock indicator
        "transparency_level_norm",          # transparency main effect (normalised 0–1)
    ]

    control_cols = [
        "task_complexity_index",
        "demand_volatility",
        "supply_disruption_count",
        "forecast_accuracy_mape",
        "decision_latency_avg",
        "target_difficulty",
        "performance_pressure_index",
        "recent_negative_shock",
    ]

    df = df.dropna(subset=emission_cols + transition_cols + control_cols)

    Y_list, X_list, Z_list = [], [], []
    ids, periods = [], []

    for mid, g in df.groupby("manager_id"):
        g = g.sort_values("period_id")
        Y = g[emission_cols].to_numpy(float)
        X = g[transition_cols].to_numpy(float)
        Z = g[control_cols].to_numpy(float)
        if len(Y) < 3:
            continue
        Y_list.append(Y)
        X_list.append(X)
        Z_list.append(Z)
        ids.append(mid)
        periods.append(g["period_id"].to_numpy())

    y_scaler = StandardScaler().fit(np.vstack(Y_list))
    x_scaler = StandardScaler().fit(np.vstack(X_list))
    z_scaler = StandardScaler().fit(np.vstack(Z_list))

    Y_list = [y_scaler.transform(y) for y in Y_list]
    X_list = [x_scaler.transform(x) for x in X_list]
    Z_list = [z_scaler.transform(z) for z in Z_list]

    return HMMData(Y_list, X_list, Z_list, ids, periods,
                   y_scaler, x_scaler, z_scaler)


# ---- Load & inspect ----
data = load_sequences(DATA_PATH)

seq_lens = [len(y) for y in data.Y]
print(f"Managers loaded : {len(data.Y)}")
print(f"Total observations: {sum(seq_lens)}")
print(f"Sequence lengths : min={min(seq_lens)}, median={int(np.median(seq_lens))}, max={max(seq_lens)}")
print(f"Emission dims (D) : {data.Y[0].shape[1]}")
print(f"Trans. covars (P) : {data.X[0].shape[1]}  "
      f"(3 benchmarks + 1 transparency main effect = 4)")
print(f"Controls (K)      : {data.Z[0].shape[1]}")


  Dropped latent truth columns to proceed with analysis data.
  share_authority_esc computed from decision_episode (mean=0.1853, std=0.0690)
Managers loaded : 120
Total observations: 3000
Sequence lengths : min=25, median=25, max=25
Emission dims (D) : 2
Trans. covars (P) : 4  (3 benchmarks + 1 transparency main effect = 4)
Controls (K)      : 8


<a id="2-hidden-markov-model-specification"></a>
## 2. Hidden Markov Model Specification
<a id="21-model-structure--22-forwardbackward-algorithm"></a>
### 2.1 Model Structure & 2.2 Forward–Backward Algorithm

In [5]:
# ============================================================
# Pre-stack all sequences to 3-D tensors for batched estimation
# ============================================================
import numpy as np

Y_stack = np.stack(data.Y)   # (N, T, D)
X_stack = np.stack(data.X)   # (N, T, P)
Z_stack = np.stack(data.Z)   # (N, T, K)

N, T, D = Y_stack.shape
P = X_stack.shape[2]
K = Z_stack.shape[2]
n_obs_total = N * T

print(f"Stacked arrays:")
print(f"  Y_stack: {Y_stack.shape}  (N={N}, T={T}, D={D})")
print(f"  X_stack: {X_stack.shape}  (N={N}, T={T}, P={P})")
print(f"  Z_stack: {Z_stack.shape}  (N={N}, T={T}, K={K})")
print(f"  n_obs_total = {n_obs_total}")

Stacked arrays:
  Y_stack: (120, 25, 2)  (N=120, T=25, D=2)
  X_stack: (120, 25, 4)  (N=120, T=25, P=4)
  Z_stack: (120, 25, 8)  (N=120, T=25, K=8)
  n_obs_total = 3000


In [6]:
# ── X collinearity check ─────────────────────────────────────────────────────
transition_cols_check = [
    "within_unit_temporal_benchmark",
    "horizontal_peer_benchmark",
    "threshold_benchmark",
    "transparency_level_norm",
]

X_flat = X_stack.reshape(-1, X_stack.shape[-1])   # (N*T, 4)
corr = np.corrcoef(X_flat.T)
df_corr = pd.DataFrame(corr, index=transition_cols_check, columns=transition_cols_check)

def color_high(val):
    return "background-color: #ff4444; color: white" if abs(val) >= 0.80 and abs(val) < 1.0 else ""

display(df_corr.round(3).style.map(color_high))

print("\nPairs with |r| >= 0.70:")
found = False
for i in range(len(transition_cols_check)):
    for j in range(i+1, len(transition_cols_check)):
        r = corr[i, j]
        if abs(r) >= 0.70:
            print(f"  {transition_cols_check[i]:40s}  {transition_cols_check[j]:40s}  r={r:.3f}")
            found = True
if not found:
    print("  None — all pairs below 0.70 ✓")


,within_unit_temporal_benchmark,horizontal_peer_benchmark,threshold_benchmark,transparency_level_norm
within_unit_temporal_benchmark,1.000000,0.680000,-0.006000,-0.007000
horizontal_peer_benchmark,0.680000,1.000000,0.003000,-0.000000
threshold_benchmark,-0.006000,0.003000,1.000000,0.028000
transparency_level_norm,-0.007000,-0.000000,0.028000,1.000000



Pairs with |r| >= 0.70:
  None — all pairs below 0.70 ✓


In [ ]:
# ============================================================
# 4. Parameters + Forward–Backward  (VECTORIZED)
# ============================================================

@dataclass
class Params:
    logit_pi: np.ndarray   # (J,)
    alpha: np.ndarray      # (J, J)
    beta: np.ndarray       # (J, J, P)
    mu: np.ndarray         # (J, D)
    W: np.ndarray          # (J, D, K)
    log_sigma: np.ndarray  # (J, D)


def _precompute(p, Y, X, Z):
    """Shared emission + transition pre-computation."""
    T, D = Y.shape
    J = p.mu.shape[0]
    means = p.mu[None, :, :] + np.einsum('jdk,tk->tjd', p.W, Z)
    residuals = Y[:, None, :] - means
    sigma2 = np.exp(2 * p.log_sigma)
    log_norm = np.sum(np.log(2 * np.pi * sigma2), axis=1)
    logB = -0.5 * (log_norm[None, :] +
                   np.sum(residuals ** 2 / sigma2[None, :, :], axis=2))
    logits_all = (p.alpha[None, :, :]
                  + np.einsum('ijp,tp->tij', p.beta, X))
    logQ_all = log_softmax(logits_all, axis=2)
    return T, J, logB, logQ_all


def forward_only(p, Y, X, Z):
    """Forward pass only — returns log-likelihood (no posterior). ~2× faster."""
    T, J, logB, logQ_all = _precompute(p, Y, X, Z)
    pi = softmax(p.logit_pi)
    log_alpha = np.empty((T, J))
    log_alpha[0] = np.log(pi) + logB[0]
    for t in range(1, T):
        log_alpha[t] = logB[t] + logsumexp(
            log_alpha[t - 1, :, None] + logQ_all[t], axis=0)
    return float(logsumexp(log_alpha[-1]))


def forward_backward(p, Y, X, Z):
    """Full forward–backward returning (ll, log_gamma)."""
    T, J, logB, logQ_all = _precompute(p, Y, X, Z)
    pi = softmax(p.logit_pi)
    log_alpha = np.empty((T, J))
    log_alpha[0] = np.log(pi) + logB[0]
    for t in range(1, T):
        log_alpha[t] = logB[t] + logsumexp(
            log_alpha[t - 1, :, None] + logQ_all[t], axis=0)
    ll = logsumexp(log_alpha[-1])

    log_beta = np.zeros((T, J))
    for t in reversed(range(T - 1)):
        log_beta[t] = logsumexp(
            logQ_all[t + 1] + logB[t + 1][None, :] + log_beta[t + 1][None, :],
            axis=1)

    log_gamma = log_alpha + log_beta
    log_gamma -= logsumexp(log_gamma, axis=1, keepdims=True)
    return ll, log_gamma

print("Section 4 — Params, forward_only() & forward_backward() defined.")

Section 4 — Params, forward_only() & forward_backward() defined.


<a id="22-maximum-likelihood-estimation"></a>
### 2.2 Maximum Likelihood Estimation
Estimate models with 2–4 latent states via maximum likelihood using L-BFGS-B optimization. To mitigate local optima, we employ multiple random initializations and diagonal-biased transition logits. Model selection follows a two-stage procedure: a computationally efficient screening phase identifies the most promising state specification using BIC, followed by a high-precision refit of the selected model using extended iterations and warm-start initialization. The final model is chosen based on the lowest Bayesian Information Criterion.

In [8]:
# ============================================================
# 5. Batched MLE Estimator  (fit_model_batched)
# ============================================================
# Accepts Y_stack, X_stack, Z_stack as EXPLICIT parameters so
# that Stack dimensions are always consistent with what was passed.
# Supports do_emission_only_warmstart for a two-phase init.
# ============================================================

import time
import numpy as np
from scipy.optimize import minimize
from scipy.special import logsumexp, log_softmax


def fit_model_batched(
    J: int,
    Y_stack: np.ndarray,
    X_stack: np.ndarray,
    Z_stack: np.ndarray,
    *,
    maxiter: int = 600,
    n_starts: int = 5,
    seed: int = 7,
    sigma_min: float = 0.05,
    sigma_max: float = 5.0,
    time_cap_min: int = 15,
    print_every: int = 50,
    l2: float = 1e-4,
    diag_bias: float = 2.0,
    maxfun: int = 200_000,
    ftol: float = 1e-8,
    gtol: float = 5e-6,
    warm_starts: list | None = None,
    use_subset: bool = False,
    subset_size: int = 80,
    do_emission_only_warmstart: bool = True,
    emission_only_maxiter: int = 120,
    emission_only_maxfun: int = 60_000,
):
    """
    Batched NH-HMM fit with Gaussian emissions and covariate-dependent
    transitions via L-BFGS-B.

    Parameters
    ----------
    Y_stack : (N, T, D) emission data
    X_stack : (N, T, P) transition covariates
    Z_stack : (N, T, K) emission controls
    do_emission_only_warmstart : if True, first optimize emission params only
        (mu, W, log_sigma) with transitions fixed, then use as init.
    """
    # Derive dimensions from passed stacks
    N_full, T, D = Y_stack.shape
    P = X_stack.shape[2]
    K = Z_stack.shape[2]

    # Optional subset for speed
    if use_subset and N_full > subset_size:
        rng_sub = np.random.default_rng(seed)
        idx = rng_sub.choice(N_full, size=subset_size, replace=False)
        Y_use = Y_stack[idx]
        X_use = X_stack[idx]
        Z_use = Z_stack[idx]
        N_use = subset_size
    else:
        Y_use, X_use, Z_use = Y_stack, X_stack, Z_stack
        N_use = N_full

    # ── pack / unpack ──
    def pack(p):
        return np.concatenate([
            p.logit_pi.ravel(), p.alpha.ravel(), p.beta.ravel(),
            p.mu.ravel(), p.W.ravel(), p.log_sigma.ravel(),
        ])

    def unpack(theta):
        idx = 0
        def take(n):
            nonlocal idx; v = theta[idx:idx+n]; idx += n; return v
        return Params(
            logit_pi=take(J),
            alpha=take(J*J).reshape(J,J),
            beta=take(J*J*P).reshape(J,J,P),
            mu=take(J*D).reshape(J,D),
            W=take(J*D*K).reshape(J,D,K),
            log_sigma=take(J*D).reshape(J,D),
        )

    # ── bounds (log_sigma only) ──
    n_logit_pi = J
    n_alpha = J*J
    n_beta = J*J*P
    n_mu = J*D
    n_W = J*D*K
    n_log_sigma = J*D
    log_sigma_start = n_logit_pi + n_alpha + n_beta + n_mu + n_W
    log_sigma_end = log_sigma_start + n_log_sigma
    total_params = log_sigma_end

    LOW, HIGH = np.log(sigma_min), np.log(sigma_max)
    bounds = [(None, None)] * total_params
    for i in range(log_sigma_start, log_sigma_end):
        bounds[i] = (LOW, HIGH)

    # ── smart init ──
    Y_flat = Y_use.reshape(-1, D)
    y_mean = Y_flat.mean(axis=0)
    y_std = np.maximum(Y_flat.std(axis=0), 1e-3)

    def smart_init_params(rng):
        mu0 = y_mean[None,:] + rng.normal(0, 1.0, (J,D)) * y_std[None,:]
        log_sigma0 = np.log(np.clip(y_std, sigma_min, sigma_max))[None,:]
        log_sigma0 = np.repeat(log_sigma0, J, axis=0)
        log_sigma0 = np.clip(log_sigma0 + rng.normal(0, 0.12, (J,D)), LOW, HIGH)
        logit_pi0 = rng.normal(0, 0.2, J)
        alpha0 = rng.normal(0, 0.20, (J,J)) + np.eye(J) * diag_bias
        beta0 = rng.normal(0, 0.02, (J,J,P))
        W0 = rng.normal(0, 0.03, (J,D,K))
        return Params(logit_pi=logit_pi0, alpha=alpha0, beta=beta0,
                      mu=mu0, W=W0, log_sigma=log_sigma0)

    # ── neg-LL (batched forward algorithm) ──
    stop_flag = {"stop": False}

    def neg_ll(theta):
        if stop_flag["stop"]:
            return 1e50
        p = unpack(theta)
        log_pi = log_softmax(p.logit_pi, axis=0)
        means = p.mu[None,None,:,:] + np.einsum("jdk,ntk->ntjd", p.W, Z_use)
        resid = Y_use[:,:,None,:] - means
        sigma2 = np.maximum(np.exp(2.0 * p.log_sigma), 1e-6)
        log_norm = np.sum(np.log(2*np.pi * sigma2), axis=1)
        logB = -0.5 * (log_norm[None,None,:] +
                       np.sum(resid**2 / sigma2[None,None,:,:], axis=3))
        logQ = log_softmax(
            p.alpha[None,None,:,:] + np.einsum("ijp,ntp->ntij", p.beta, X_use),
            axis=3)
        la = log_pi[None,:] + logB[:,0,:]
        for t in range(1, T):
            la = logB[:,t,:] + logsumexp(la[:,:,None] + logQ[:,t,:,:], axis=1)
        ll = np.sum(logsumexp(la, axis=1))
        return -float(ll) if np.isfinite(ll) else 1e40

    def objective(theta):
        base = neg_ll(theta)
        if not np.isfinite(base) or l2 <= 0:
            return base if np.isfinite(base) else 1e40
        p = unpack(theta)
        pen = (np.sum(p.alpha**2) + np.sum(p.beta**2) +
               np.sum(p.W**2) + 0.10*np.sum(p.mu**2))
        return base + l2 * pen

    # ── emission-only warmstart objective ──
    def emission_only_objective(em_theta, fixed_logit_pi, fixed_alpha, fixed_beta):
        """Optimize only mu, W, log_sigma with transitions frozen."""
        idx = 0
        def take(n):
            nonlocal idx; v = em_theta[idx:idx+n]; idx += n; return v
        mu = take(J*D).reshape(J,D)
        W = take(J*D*K).reshape(J,D,K)
        log_sigma = take(J*D).reshape(J,D)
        p = Params(logit_pi=fixed_logit_pi, alpha=fixed_alpha,
                   beta=fixed_beta, mu=mu, W=W, log_sigma=log_sigma)
        theta_full = pack(p)
        return objective(theta_full)

    # ── multi-start optimization ──
    init_list = list(warm_starts) if warm_starts else []
    runs = []

    for s in range(n_starts):
        rng = np.random.default_rng(seed + s)
        stop_flag["stop"] = False

        # Initialize
        if s < len(init_list):
            p0 = init_list[s]
            # Ensure dimensions match
            if p0.W.shape != (J, D, K):
                print(f"  ⚠️  start {s+1}: warm start W shape {p0.W.shape} "
                      f"!= ({J},{D},{K}), using random init")
                p0 = smart_init_params(rng)
            else:
                p0 = Params(
                    logit_pi=p0.logit_pi + rng.normal(0, 0.03, J),
                    alpha=p0.alpha + rng.normal(0, 0.03, (J,J)),
                    beta=p0.beta + rng.normal(0, 0.008, (J,J,P)),
                    mu=p0.mu + rng.normal(0, 0.05, (J,D)),
                    W=p0.W + rng.normal(0, 0.01, (J,D,K)),
                    log_sigma=np.clip(p0.log_sigma + rng.normal(0, 0.02, (J,D)),
                                      LOW, HIGH),
                )
        else:
            p0 = smart_init_params(rng)

        # Phase 1 (optional): emission-only warmstart
        if do_emission_only_warmstart and s >= len(init_list):
            em_theta0 = np.concatenate([
                p0.mu.ravel(), p0.W.ravel(), p0.log_sigma.ravel()])
            em_bounds = ([(None,None)]*(J*D + J*D*K) +
                         [(LOW,HIGH)]*(J*D))
            em_res = minimize(
                emission_only_objective, em_theta0,
                args=(p0.logit_pi, p0.alpha, p0.beta),
                method="L-BFGS-B", bounds=em_bounds,
                options={"maxiter": emission_only_maxiter,
                         "maxfun": emission_only_maxfun})
            # unpack emission params back
            eidx = 0
            def etake(n):
                nonlocal eidx; v = em_res.x[eidx:eidx+n]; eidx += n; return v
            p0 = Params(logit_pi=p0.logit_pi, alpha=p0.alpha,
                        beta=p0.beta,
                        mu=etake(J*D).reshape(J,D),
                        W=etake(J*D*K).reshape(J,D,K),
                        log_sigma=etake(J*D).reshape(J,D))

        # Phase 2: full optimization
        theta0 = pack(p0)
        start_time = time.time()
        iter_counter = {"i": 0}

        def callback(_xk):
            iter_counter["i"] += 1
            if iter_counter["i"] % print_every == 0:
                elapsed_min = (time.time() - start_time) / 60
                print(f"    J={J} start {s+1}/{n_starts} "
                      f"iter={iter_counter['i']} elapsed={elapsed_min:.1f} min",
                      flush=True)
            if (time.time() - start_time) > time_cap_min * 60:
                stop_flag["stop"] = True

        res = minimize(objective, theta0, method="L-BFGS-B",
                       bounds=bounds, callback=callback,
                       options={"maxiter": maxiter, "maxfun": maxfun,
                                "ftol": ftol, "gtol": gtol})

        if stop_flag["stop"]:
            res.success = False
            res.message = f"Time cap reached ({time_cap_min} min)"

        true_negll = neg_ll(res.x)
        runs.append((unpack(res.x), res, true_negll))
        print(f"    done: J={J} start {s+1}/{n_starts} success={res.success} "
              f"nit={getattr(res,'nit',None)} true_negLL={true_negll:.2f} "
              f"msg={res.message}", flush=True)

    # ── pick best run ──
    converged = [(p,r,tnl) for (p,r,tnl) in runs if bool(r.success)]
    if converged:
        best_p, best_res, best_tnl = min(converged, key=lambda t: t[2])
        best_is_conv = True
    else:
        best_p, best_res, best_tnl = min(runs, key=lambda t: t[2])
        best_is_conv = False

    best_res.true_negll = best_tnl
    best_res.true_ll = -best_tnl
    best_res.k_params = len(best_res.x)
    return best_p, best_res, best_is_conv


print("fit_model_batched() defined — accepts Y_stack, X_stack, Z_stack explicitly.")

fit_model_batched() defined — accepts Y_stack, X_stack, Z_stack explicitly.


In [9]:
# ============================================================
# STAGE 1: SCREENING  (J = 2, 3, 4)
# Goal:
#   1) Subset warm-start for each J (Stage 1A)
#   2) Full-data fit for each J using warm start (Stage 1B)
#   3) Select best J by BIC
# ============================================================

import time
import numpy as np
import pandas as pd

# ----------------------------
# Helper: build configs by J
# ----------------------------
def make_cfg_stage1a(J: int) -> dict:
    """Subset screening configs to generate warm start."""
    if J == 2:
        return dict(
            maxiter=600,
            n_starts=6,
            time_cap_min=15,
            diag_bias=2.0,
            maxfun=220000,
            use_subset=False,
            subset_size=80,
            l2=0.01,
            ftol=1e-7,
            gtol=1e-5,
            do_emission_only_warmstart=True,
            emission_only_maxiter=140,
        )
    if J == 3:
        return dict(
            maxiter=900,
            n_starts=10,
            time_cap_min=25,
            diag_bias=2.6,
            maxfun=380000,
            use_subset=True,
            subset_size=95,
            l2=0.01,
            ftol=1e-7,
            gtol=2e-5,
            do_emission_only_warmstart=True,
            emission_only_maxiter=180,
        )
    # J == 4  — raised caps for convergence with D=2 model
    return dict(
        maxiter=1500,
        n_starts=14,
        time_cap_min=60,
        diag_bias=2.9,
        maxfun=800000,
        use_subset=True,
        subset_size=95,
        l2=0.01,
        ftol=1e-7,
        gtol=2e-5,
        do_emission_only_warmstart=True,
        emission_only_maxiter=260,
    )


def make_cfg_stage1b(J: int) -> dict:
    """Full-data screening configs seeded with subset warm starts."""
    if J == 2:
        return dict(
            maxiter=700,
            n_starts=6,
            time_cap_min=18,
            diag_bias=2.0,
            maxfun=260000,
            use_subset=False,
            l2=0.01,
            ftol=1e-7,
            gtol=1e-5,
            do_emission_only_warmstart=True,
            emission_only_maxiter=140,
        )
    if J == 3:
        return dict(
            maxiter=1200,
            n_starts=10,
            time_cap_min=32,
            diag_bias=2.4,
            maxfun=520000,
            use_subset=False,
            l2=0.01,
            ftol=1e-7,
            gtol=2e-5,
            do_emission_only_warmstart=True,
            emission_only_maxiter=200,
        )
    # J == 4  — raised caps for convergence with D=2 model
    return dict(
        maxiter=2000,
        n_starts=14,
        time_cap_min=80,
        diag_bias=2.7,
        maxfun=1000000,
        use_subset=False,
        l2=0.01,
        ftol=1e-7,
        gtol=2e-5,
        do_emission_only_warmstart=True,
        emission_only_maxiter=320,
    )


# ----------------------------
# Stage 1A: Subset screening  (J = 2, 3, 4)
# ----------------------------
warm_by_J: dict[int, object] = {}
subset_log = []

print("\n=== STAGE 1A: Subset Screening — J = 2, 3, 4 ===")

for J in [2, 3, 4]:
    cfg = make_cfg_stage1a(J)

    print(f"\n[SubsetScreen] Fitting J={J} ...", flush=True)
    t0 = time.time()

    p_hat_sub, res_sub, is_conv_sub = fit_model_batched(
        J=J,
        Y_stack=Y_stack, X_stack=X_stack, Z_stack=Z_stack,
        seed=7,
        sigma_min=0.1, sigma_max=3.5,
        print_every=50,
        **cfg
    )

    elapsed = time.time() - t0
    warm_by_J[J] = p_hat_sub

    subset_log.append({
        "stage": "subset",
        "J": J,
        "soft_converged": bool(is_conv_sub),
        "scipy_success": bool(getattr(res_sub, "success", False)),
        "LL": float(getattr(res_sub, "true_ll", np.nan)),
        "time_s": elapsed,
        "message": str(getattr(res_sub, "message", "")),
    })

    print(
        f"[SubsetScreen] J={J} soft_converged={bool(is_conv_sub)} "
        f"LL={float(getattr(res_sub,'true_ll',np.nan)):.1f} "
        f"({elapsed/60:.1f} min) msg={getattr(res_sub,'message','')}",
        flush=True
    )

df_subset = pd.DataFrame(subset_log)
display(df_subset.round(2))


# ----------------------------
# Stage 1B: Full-data — J = 2, 3, 4
# ----------------------------
screen_results = []

print("\n=== STAGE 1B: Full-data Fit — J = 2, 3, 4 (warm-started) ===")

for J in [2, 3, 4]:
    cfg_full = make_cfg_stage1b(J)

    warm = warm_by_J.get(J)
    if warm is None:
        print(f"⚠️  No warm start for J={J}. Skipping full-data screening.")
        continue

    print(f"\n[ScreenFull] Fitting J={J} (warm-started) ...", flush=True)
    t0 = time.time()

    p_hat, res, is_conv = fit_model_batched(
        J=J,
        Y_stack=Y_stack, X_stack=X_stack, Z_stack=Z_stack,
        seed=77,
        sigma_min=0.1, sigma_max=3.5,
        print_every=50,
        warm_starts=[warm],
        **cfg_full
    )

    elapsed = time.time() - t0

    ll_total = float(getattr(res, "true_ll", np.nan))
    k_params = len(getattr(res, "x", []))

    bic = np.log(n_obs_total) * k_params - 2.0 * ll_total
    aic = 2.0 * k_params - 2.0 * ll_total

    screen_results.append({
        "stage": "screen_full",
        "J": J,
        "params": k_params,
        "LL": ll_total,
        "AIC": aic,
        "BIC": bic,
        "converged_soft": bool(is_conv),
        "converged_scipy": bool(getattr(res, "success", False)),
        "time_s": elapsed,
        "message": str(getattr(res, "message", "")),
    })

    sym = "✓" if bool(is_conv) else "✗"
    print(
        f"[ScreenFull] J={J} LL={ll_total:.1f} BIC={bic:.1f} params={k_params} "
        f"{sym} ({elapsed/60:.1f} min)"
    )
    print(f"            message: {getattr(res,'message','')}")

df_screen = pd.DataFrame(screen_results)
display(df_screen.round(2))


# ----------------------------
# Select best J by BIC
# ----------------------------
best_J_screen = int(df_screen.loc[df_screen["BIC"].idxmin(), "J"])
best_row = df_screen[df_screen["J"] == best_J_screen].iloc[0]
print(
    f"\n★ Best J by BIC: J={best_J_screen} "
    f"(BIC={best_row['BIC']:.1f}, soft_converged={bool(best_row['converged_soft'])})"
)
print(df_screen[["J", "params", "LL", "AIC", "BIC", "converged_soft"]].to_string(index=False))



=== STAGE 1A: Subset Screening — J = 2, 3, 4 ===

[SubsetScreen] Fitting J=2 ...
    J=2 start 1/6 iter=50 elapsed=0.4 min
    J=2 start 1/6 iter=100 elapsed=0.8 min
    J=2 start 1/6 iter=150 elapsed=1.1 min
    done: J=2 start 1/6 success=True nit=190 true_negLL=2908.00 msg=CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
    J=2 start 2/6 iter=50 elapsed=0.4 min
    J=2 start 2/6 iter=100 elapsed=0.8 min
    J=2 start 2/6 iter=150 elapsed=1.2 min
    J=2 start 2/6 iter=200 elapsed=1.6 min
    done: J=2 start 2/6 success=True nit=215 true_negLL=2908.00 msg=CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
    J=2 start 3/6 iter=50 elapsed=0.5 min
    J=2 start 3/6 iter=100 elapsed=0.9 min


KeyboardInterrupt: 

#### Final refit with best J!!!

In [ ]:
# ============================================================
# STAGE 2: FINAL REFIT (robust version)
# ============================================================

import time
import numpy as np

# ----------------------------
# Select best J from Stage 1
# ----------------------------
if "best_J" in globals():
    BEST_J = int(best_J)
elif "best_J_screen" in globals():
    BEST_J = int(best_J_screen)
else:
    raise NameError("Run Stage 1 screening first.")

print(f"\n=== STAGE 2: FINAL REFIT (BEST J={BEST_J}) ===")

# ----------------------------
# Warm start
# Skip any warm start whose W shape doesn't match Z_stack K dimension
# ----------------------------
warm_list = []
K_now = Z_stack.shape[2]

for src_name, src_dict in [
    ("best_params_full_by_J", globals().get("best_params_full_by_J", {})),
    ("warm_by_J", globals().get("warm_by_J", {})),
]:
    if isinstance(src_dict, dict) and BEST_J in src_dict:
        ws = src_dict[BEST_J]
        if hasattr(ws, "W") and ws.W.shape[2] == K_now:
            warm_list.append(ws)
            print(f"✓ Using warm start from {src_name} (K={ws.W.shape[2]})")
            break
        else:
            ws_k = ws.W.shape[2] if hasattr(ws, "W") else "?"
            print(f"⚠️ Skipping warm start from {src_name} (K={ws_k} ≠ {K_now})")

if not warm_list:
    print("⚠️ No compatible warm start found — random initialization")

# ----------------------------
# Stronger final configuration
# NOTE: l2=0.01 prevents degenerate absorbing transitions
#       Caps scale with J: J=4 needs more iterations and time to converge.
# ----------------------------
final_cfg = dict(
    maxiter=1600 if BEST_J <= 3 else 2200,
    n_starts=12 if BEST_J <= 3 else 15,
    seed=777,
    time_cap_min=45 if BEST_J <= 3 else 90,
    diag_bias=2.4 if BEST_J <= 3 else 2.7,
    maxfun=900_000 if BEST_J <= 3 else 1_300_000,
    use_subset=False,
    l2=0.01,
    do_emission_only_warmstart=True,
    emission_only_maxiter=300 if BEST_J <= 3 else 400,
    ftol=1e-9,
    gtol=1e-6,
)

# ----------------------------
# Run final estimation
# ----------------------------
t0 = time.time()

best_p_final, best_res_final, best_is_conv_final = fit_model_batched(
    J=BEST_J,
    Y_stack=Y_stack,
    X_stack=X_stack,
    Z_stack=Z_stack,
    sigma_min=0.1,
    sigma_max=3.5,
    print_every=50,
    warm_starts=warm_list,
    **final_cfg
)

elapsed = time.time() - t0

# ----------------------------
# Metrics
# ----------------------------
ll_total = float(getattr(best_res_final, "true_ll", np.nan))
k_params = len(getattr(best_res_final, "x", []))
bic = np.log(n_obs_total) * k_params - 2.0 * ll_total
aic = 2.0 * k_params - 2.0 * ll_total

print("\n" + "="*60)
print(f"FINAL MODEL (J={BEST_J})")
print(f"LL:  {ll_total:.2f}")
print(f"AIC: {aic:.2f}")
print(f"BIC: {bic:.2f}")
print(f"Soft-converged: {bool(best_is_conv_final)}")
print(f"SciPy success:  {bool(getattr(best_res_final,'success',False))}")
print(f"Runtime: {elapsed/60:.1f} minutes")
print("="*60)

best_model = best_p_final



=== STAGE 2: FINAL REFIT (BEST J=2) ===
✓ Using warm start from warm_by_J (K=8)
    J=2 start 1/12 iter=50 elapsed=0.4 min
    J=2 start 1/12 iter=100 elapsed=0.8 min
    done: J=2 start 1/12 success=True nit=143 true_negLL=2907.86 msg=CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
    J=2 start 2/12 iter=50 elapsed=0.4 min
    J=2 start 2/12 iter=100 elapsed=0.8 min
    J=2 start 2/12 iter=150 elapsed=1.2 min
    J=2 start 2/12 iter=200 elapsed=1.7 min
    J=2 start 2/12 iter=250 elapsed=2.1 min
    J=2 start 2/12 iter=300 elapsed=2.5 min
    J=2 start 2/12 iter=350 elapsed=2.9 min
    J=2 start 2/12 iter=400 elapsed=3.3 min
    done: J=2 start 2/12 success=True nit=447 true_negLL=2907.86 msg=CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
    J=2 start 3/12 iter=50 elapsed=0.4 min
    J=2 start 3/12 iter=100 elapsed=0.9 min
    J=2 start 3/12 iter=150 elapsed=1.3 min
    J=2 start 3/12 iter=200 elapsed=1.7 min
    J=2 start 3/12 iter=250 elapsed=2.0 min
    J=2 start 3/12

In [ ]:

import pickle

# Save best model artifacts for later reuse (no refit needed)

best_J = int(best_J) if "best_J" in globals() else int(globals().get("BEST_J", best_J_screen))

model_artifacts = {
    "best_model": best_model,
    "best_J": int(best_J),
    "best_res_final": globals().get("best_res_final", None),
    "final_cfg": globals().get("final_cfg", None),
    "ll_total": globals().get("ll_total", None),
    "k_params": globals().get("k_params", None),
    "n_obs_total": globals().get("n_obs_total", None),
    "aic": globals().get("aic", None),
    "bic": globals().get("bic", None),
    "emission_cols": globals().get("emission_cols", None),
    "transition_cols": globals().get("transition_cols", None),
    "control_cols": globals().get("control_cols", None),
    "label_map": globals().get("label_map", None),
    "state_order_1idx": globals().get("state_order_1idx", None),
    "y_scaler": getattr(data, "y_scaler", None),
    "x_scaler": getattr(data, "x_scaler", None),
    "z_scaler": getattr(data, "z_scaler", None),
}

out_path = Path("best_model_artifacts_v3.pkl")
with out_path.open("wb") as f:
    pickle.dump(model_artifacts, f)

print(f"Saved best model artifacts → {out_path.resolve()}")


Saved best model artifacts → C:\Users\Admin\OneDrive\Desktop\Algorithm-Appreciation-and-Aversion-in-Triadic-Delegation-Settings\data_analysis\best_model_artifacts_v2.pkl


<a id="3-model-selection-and-parameter-estimation"></a>
## 3. Model Selection and Parameter Estimation


In [ ]:
# ============================================================
# 6. Inspect Estimated Parameters  (UPDATED + REPORTABLE)
#   - Uses Stage 2 final model handles
#   - Adds: posterior occupancy + certainty (from gamma if available)
#   - Prints: transition probabilities (A) in addition to intercept logits (alpha)
#   - Keeps: emissions on original scale
#   - D=2: ai_decision_authority_share, share_authority_esc
# ============================================================

import numpy as np
import pandas as pd
from scipy.special import softmax

# --- Use Stage 2 final outputs ---
best_model = best_p_final
best_J = int(best_J_final) if "best_J_final" in globals() else int(BEST_J)

# D=2: override_rate removed (= 1 − ai_decision_authority_share by construction)
emission_cols = ["ai_decision_authority_share", "share_authority_esc"]

# ----------------------------
# Emissions (original scale)
# ----------------------------
mu_orig = data.y_scaler.inverse_transform(best_model.mu)                  # (J,D)
sigma_orig = np.exp(best_model.log_sigma) * data.y_scaler.scale_          # (J,D)

print(f"Best model: J={best_J} hidden states, D={len(emission_cols)} emissions\n")

em_rows = []
for j in range(best_J):
    for d, col in enumerate(emission_cols):
        em_rows.append({
            "State": j + 1,
            "Variable": col,
            "Mean (orig scale)": float(mu_orig[j, d]),
            "Std (orig scale)": float(sigma_orig[j, d]),
        })

em_df = pd.DataFrame(em_rows)
em_df["Mean (orig scale)"] = em_df["Mean (orig scale)"].round(4)
em_df["Std (orig scale)"]  = em_df["Std (orig scale)"].round(4)

print("── Emission Parameters (original scale) ──")
display(em_df)

# ----------------------------
# Initial state distribution π
# ----------------------------
pi = softmax(best_model.logit_pi)   # (J,)
print("\n── Initial State Distribution (π) ──")
for j in range(best_J):
    print(f"  π(state {j+1}) = {pi[j]:.4f}")

# ----------------------------
# Transition intercepts (logit scale) α
# ----------------------------
print("\n── Transition Intercept Matrix (α logits) ──")
alpha_df = pd.DataFrame(
    np.round(best_model.alpha, 3),
    index=[f"from {j+1}" for j in range(best_J)],
    columns=[f"to {j+1}" for j in range(best_J)]
)
display(alpha_df)

# ----------------------------
# Average transition probabilities (full-sample)
# NOTE: transitions depend on X via beta. We compute the mean transition matrix
# across all (i,t) using the mean X over (N, T) — excluding t=0 where .diff(1)
# produces NaN, and using t=1..T-1 which matches the forward-pass usage.
# ----------------------------
X_mean = np.nanmean(X_stack[:, 1:, :], axis=(0, 1))  # (P,)

# logits[j,k] = alpha[j,k] + beta[j,k,:] @ X_mean
logits_mean = best_model.alpha + np.tensordot(best_model.beta, X_mean, axes=([2], [0]))  # (J,J)
A_mean = softmax(logits_mean, axis=1)  # row-normalize over "to" states

print("\n── Mean Transition Matrix (A) at mean X ──")
A_df = pd.DataFrame(
    np.round(A_mean, 4),
    index=[f"from {j+1}" for j in range(best_J)],
    columns=[f"to {j+1}" for j in range(best_J)]
)
display(A_df)

# ----------------------------
# Posterior diagnostics (compute fresh from forward-backward)
# ----------------------------
print("\n── Posterior Diagnostics (from gamma) ──")
all_gamma = []
for Y_i, X_i, Z_i in zip(data.Y, data.X, data.Z):
    _, log_g = forward_backward(best_model, Y_i, X_i, Z_i)
    all_gamma.append(np.exp(log_g))

gamma_all = np.concatenate(all_gamma, axis=0)  # (N_total, J)
occupancy = gamma_all.mean(axis=0)              # (J,)
certainty = gamma_all.max(axis=1).mean()        # scalar

for j in range(best_J):
    print(f"  State {j+1} occupancy: {occupancy[j]*100:.2f}%")
print(f"  Mean posterior certainty: {certainty:.3f}")


Best model: J=2 hidden states, D=2 emissions

── Emission Parameters (original scale) ──


,State,Variable,Mean (orig scale),Std (orig scale)
0,1,ai_decision_authority_share,0.4326,0.0509
1,1,share_authority_esc,0.2213,0.0500
2,2,ai_decision_authority_share,0.3964,0.0500
3,2,share_authority_esc,0.1458,0.0429



── Initial State Distribution (π) ──
  π(state 1) = 0.7078
  π(state 2) = 0.2922

── Transition Intercept Matrix (α logits) ──


,to 1,to 2
from 1,1.216,1.241
from 2,1.122,1.178



── Mean Transition Matrix (A) at mean X ──


,to 1,to 2
from 1,0.4786,0.5214
from 2,0.4755,0.5245



── Posterior Diagnostics (from gamma) ──
  State 1 occupancy: 50.09%
  State 2 occupancy: 49.91%
  Mean posterior certainty: 0.832
